# Laboratory Exercise: Image Classification Using Pre-trained Models

## Celebrity Faces Classification Using PyTorch and Torchvision

**Platform:** Google Colab  
**Framework:** PyTorch  
**Dataset:** Celebrity Faces Dataset  
**Task:** Multi-class image classification  
**Models:** ResNet18, ResNet50, DenseNet121, MobileNetV3-Large, EfficientNet-B0

execute:
  execute_notebooks: "off"

## Introduction

Image classification assigns images to predefined classes. In this activity, a Celebrity Faces dataset is used to evaluate five pre-trained models from torchvision.models: ResNet18, ResNet50, DenseNet121, MobileNetV3-Large, and EfficientNet-B0. Their performance is compared using accuracy, precision, recall, and F1-score.

## Import Libraries

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np

import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from torchvision import (
    transforms,
    models
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from tqdm.auto import tqdm

print("Libraries imported successfully.")

The required libraries were successfully imported. These libraries provide the tools needed for dataset handling, image preprocessing, deep learning model inference, visualization, and performance evaluation.

## Locating ZIP file

In [ ]:
zip_files = [
    file
    for file in os.listdir("/content")
    if file.lower().endswith(".zip")
]

print("ZIP files found:")

for file in zip_files:
    print("-", file)

In [ ]:
if len(zip_files) == 0:
    raise FileNotFoundError(
        "No ZIP file was found in /content. "
        "Please check that the dataset is uploaded in the current Colab session."
    )

ZIP_FILE = os.path.join(
    "/content",
    zip_files[0]
)

print("Using dataset:")
print(ZIP_FILE)

The output confirms that the Celebrity Faces dataset ZIP file is available in the current Google Colab environment. This file will be extracted and used as the source of the image dataset.

## Extracting the Dataset

In [ ]:
EXTRACT_PATH = "/content/celebrity_dataset"

os.makedirs(
    EXTRACT_PATH,
    exist_ok=True
)

with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as zip_ref:

    zip_ref.extractall(
        EXTRACT_PATH
    )

print("Dataset extracted successfully!")
print("Location:", EXTRACT_PATH)

The dataset was successfully extracted from the ZIP file. The extracted directory contains the image files organized into folders representing the different celebrity classes.

## Inspecting the Dataset Structure

In [ ]:
for root_dir, dirs, files in os.walk(
    EXTRACT_PATH
):

    level = root_dir.replace(
        EXTRACT_PATH,
        ""
    ).count(os.sep)

    if level > 2:
        continue

    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root_dir)}/"
    )

    for file in files[:5]:

        print(
            f"{indent}    {file}"
        )

The directory structure shows how the dataset is organized. Each folder represents a different celebrity class, while the images inside each folder are examples belonging to that class. This folder-based organization allows the images to be assigned their corresponding labels.

## Automatically Finding the Dataset Root


In [ ]:
IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)


def contains_images(folder):

    try:
        files = os.listdir(folder)
    except:
        return False

    return any(
        file.lower().endswith(
            IMAGE_EXTENSIONS
        )
        for file in files
    )


root = None

for current_root, dirs, files in os.walk(
    EXTRACT_PATH
):

    image_class_folders = []

    for directory in dirs:

        folder_path = os.path.join(
            current_root,
            directory
        )

        if contains_images(folder_path):

            image_class_folders.append(
                directory
            )

    if len(image_class_folders) >= 2:

        root = current_root
        break


if root is None:

    raise RuntimeError(
        "Could not automatically find the celebrity class folders."
    )


print("Dataset root:")
print(root)

The dataset root identifies the directory containing the celebrity class folders. This directory is used as the starting point for collecting the image paths and their corresponding class labels.

## Identifying the Celebrity Classes

In [ ]:
classes = sorted([
    folder
    for folder in os.listdir(root)
    if os.path.isdir(
        os.path.join(root, folder)
    )
    and contains_images(
        os.path.join(root, folder)
    )
])

print(
    "Number of celebrity classes:",
    len(classes)
)

print("\nCelebrity classes:")

for index, cls in enumerate(classes):

    print(
        f"{index}: {cls}"
    )

The output shows the different celebrity classes present in the dataset. Each celebrity is assigned a numerical label, starting from 0. These numerical labels are used by the PyTorch model during training and evaluation.

## Exploring the Dataset

In [ ]:
class_counts = {}

for cls in classes:

    class_path = os.path.join(
        root,
        cls
    )

    count = sum(
        1
        for file in os.listdir(class_path)
        if file.lower().endswith(
            IMAGE_EXTENSIONS
        )
    )

    class_counts[cls] = count


class_distribution = pd.DataFrame(
    list(class_counts.items()),
    columns=[
        "Celebrity",
        "Number of Images"
    ]
)

class_distribution

## Visualize Dataset Distribution
The table shows the number of images available for each celebrity class. It provides an overview of how the images are distributed across the dataset. A relatively similar number of images across classes helps reduce potential class imbalance during model training and evaluation.

In [ ]:
plt.figure(
    figsize=(12, 6)
)

plt.bar(
    class_distribution["Celebrity"],
    class_distribution["Number of Images"]
)

plt.xlabel("Celebrity")
plt.ylabel("Number of Images")

plt.title(
    "Number of Images per Celebrity Class"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

The bar chart illustrates the number of images available for each celebrity class. The height of each bar represents the number of images belonging to that celebrity. This visualization makes it easier to identify whether the dataset is balanced or whether some celebrity classes contain substantially more images than others.

## Data Dictionary
The data dictionary contains two main components: `X`, which stores the file paths of the images, and `y`, which stores the numerical labels corresponding to each celebrity. The equal number of image paths and labels confirms that every image has an associated class label.

In [ ]:
data_dict = {
    "X": [],
    "y": []
}

class_to_idx = {
    cls: index
    for index, cls in enumerate(classes)
}


for cls in classes:

    class_path = os.path.join(
        root,
        cls
    )

    for file in os.listdir(
        class_path
    ):

        if file.lower().endswith(
            IMAGE_EXTENSIONS
        ):

            image_path = os.path.join(
                class_path,
                file
            )

            data_dict["X"].append(
                image_path
            )

            data_dict["y"].append(
                class_to_idx[cls]
            )


print(
    "Total image paths:",
    len(data_dict["X"])
)

print(
    "Total labels:",
    len(data_dict["y"])
)

## DataFrame
The table provides a preview of the curated dataset. The `X` column contains the file path of each image, while `y` contains the numerical class label. The `Celebrity` column provides the corresponding human-readable class name. This confirms that the images and their labels have been correctly organized before being passed to the PyTorch Dataset.

In [ ]:
df = pd.DataFrame(
    data_dict
)

df.head(10)

In [ ]:
df["Celebrity"] = df["y"].map(
    lambda x: classes[x]
)

df.head(10)

## Creating the Custom PyTorch Dataset
The custom PyTorch Dataset was created to control how images are loaded, transformed, and returned to the model. Each dataset item consists of an image tensor and its corresponding numerical label.

In [ ]:
class CustomDataset(Dataset):

    def __init__(
        self,
        data_dict,
        transforms=None
    ):

        self.data_dict = data_dict

        self.transforms = transforms


    def __getitem__(self, idx):

        # Image path
        image_path = self.data_dict[
            "X"
        ][idx]

        # Label
        label = self.data_dict[
            "y"
        ][idx]

        # Read image
        image = cv2.imread(
            image_path
        )

        # Convert BGR to RGB
        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # Apply transformations
        if self.transforms:

            image = self.transforms(
                image
            )

        # Convert label to tensor
        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label


    def __len__(self):

        return len(
            self.data_dict["X"]
        )

## Define Image Transformations
The images are resized to 224 × 224 pixels and converted into tensors. ImageNet normalization is also applied because the models used in this experiment were pre-trained using ImageNet data. Standardizing the input format allows the images to be processed consistently by all five models.

In [ ]:
transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

## Check Image Shapes
The image tensor has the shape `[3, 224, 224]`, where 3 represents the RGB color channels and 224 × 224 represents the standardized image dimensions. This confirms that the preprocessing step successfully converted the images into the expected input format.

In [ ]:
dataset = CustomDataset(
    data_dict=data_dict,
    transforms=transform
)

for i in range(
    min(10, len(dataset))
):

    image, label = dataset[i]

    print(
        f"Image {i}:",
        image.shape,
        "| Label:",
        label.item(),
        "| Celebrity:",
        classes[label.item()]
    )

## Display Sample Images
The figure displays sample images from the Celebrity Faces dataset together with their corresponding class labels. The visualization provides a qualitative check that the images were loaded correctly and that the assigned labels correspond to the expected celebrity classes.

In [ ]:
plt.figure(
    figsize=(12, 8)
)

for i in range(
    min(12, len(dataset))
):

    image, label = dataset[i]

    # Reverse normalization
    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(3, 1, 1)

    image_display = (
        image * std + mean
    )

    image_display = torch.clamp(
        image_display,
        0,
        1
    )

    plt.subplot(
        3,
        4,
        i + 1
    )

    plt.imshow(
        image_display.permute(
            1,
            2,
            0
        )
    )

    plt.title(
        classes[label.item()]
    )

    plt.axis("off")


plt.tight_layout()

plt.show()

## Split the Dataset
The dataset was divided into training, validation, and testing subsets. The training set is used to update the model's classification layer, the validation set is used to monitor performance during training, and the test set is reserved for the final evaluation of the trained models.


We'll use:
*   70% training
*   15% validation
*   15% testing


In [ ]:
total_size = len(dataset)

train_size = int(
    0.70 * total_size
)

validation_size = int(
    0.15 * total_size
)

test_size = (
    total_size
    - train_size
    - validation_size
)

generator = torch.Generator().manual_seed(42)

train_dataset, validation_dataset, test_dataset = random_split(
    dataset,
    [
        train_size,
        validation_size,
        test_size
    ],
    generator=generator
)

print(
    "Training:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

print(
    "Testing:",
    len(test_dataset)
)

## Create DataLoaders
The DataLoaders organize the images into batches so that they can be processed efficiently by the neural networks. The training DataLoader shuffles the data, while the validation and test DataLoaders maintain a consistent order for evaluation.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "DataLoaders created successfully."
)

## Load Five Pre-trained Models
Five different pre-trained convolutional neural network architectures were selected for comparison. These models differ in their architecture, depth, parameter count, and computational requirements. Comparing multiple architectures allows their performance on the same Celebrity Faces dataset to be evaluated.

- ResNet18
- ResNet50
- DenseNet121
- MobileNetV3-Large
- EfficientNet-B0

In [ ]:
model_list = {

    "ResNet18":
        models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        ),

    "ResNet50":
        models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        ),

    "DenseNet121":
        models.densenet121(
            weights=models.DenseNet121_Weights.DEFAULT
        ),

    "MobileNetV3-Large":
        models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.DEFAULT
        ),

    "EfficientNet-B0":
        models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
}

print("Five pretrained models loaded.")

for name in model_list:
    print("-", name)

## Adapt Models to Celebrity Classes
The original classification layers of the pre-trained models were replaced with new classification layers corresponding to the number of celebrity classes in the dataset. The pre-trained feature extraction layers were kept frozen so that the models could reuse the visual features learned from ImageNet.

In [ ]:
num_classes = len(classes)

print(
    "Number of celebrity classes:",
    num_classes
)

In [ ]:
def adapt_model(
    model,
    model_name,
    num_classes
):

    # Freeze pretrained layers
    for param in model.parameters():

        param.requires_grad = False


    # ResNet
    if model_name in [
        "ResNet18",
        "ResNet50"
    ]:

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )


    # DenseNet
    elif model_name == "DenseNet121":

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )


    # MobileNet
    elif model_name == "MobileNetV3-Large":

        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features,
            num_classes
        )


    # EfficientNet
    elif model_name == "EfficientNet-B0":

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )


    return model

## Prepare the Models

In [ ]:
adapted_models = {}

for name, model in model_list.items():

    model = adapt_model(
        model,
        name,
        num_classes
    )

    model = model.to(device)

    adapted_models[name] = model

    print(
        name,
        "is ready."
    )

## Train the Classification Layer


In [ ]:
def train_model(
    model,
    train_loader,
    validation_loader,
    epochs=5
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=0.001
    )

    best_accuracy = 0.0

    best_state = None


    for epoch in range(epochs):

        model.train()

        running_correct = 0
        running_total = 0


        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}"
        )


        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()


            predictions = outputs.argmax(
                dim=1
            )

            running_correct += (
                predictions == labels
            ).sum().item()

            running_total += (
                labels.size(0)
            )


            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )


        train_accuracy = (
            running_correct /
            running_total
        )


        # Validation
        model.eval()

        validation_correct = 0
        validation_total = 0


        with torch.no_grad():

            for images, labels in validation_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(
                    images
                )

                predictions = outputs.argmax(
                    dim=1
                )

                validation_correct += (
                    predictions == labels
                ).sum().item()

                validation_total += (
                    labels.size(0)
                )


        validation_accuracy = (
            validation_correct /
            validation_total
        )


        print(
            f"Epoch {epoch + 1}: "
            f"Train Accuracy = "
            f"{train_accuracy:.4f} | "
            f"Validation Accuracy = "
            f"{validation_accuracy:.4f}"
        )


        if validation_accuracy > best_accuracy:

            best_accuracy = validation_accuracy

            best_state = copy.deepcopy(
                model.state_dict()
            )


    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return model

## Train All Five Models

In [ ]:
trained_models = {}

for name, model in adapted_models.items():

    print("\n")
    print("=" * 70)
    print(
        f"Training: {name}"
    )
    print("=" * 70)

    trained_models[name] = train_model(
        model,
        train_loader,
        validation_loader,
        epochs=5
    )

## Model Inference

After training the new classification layers, each model is evaluated on the unseen test dataset. Predictions are compared with the actual celebrity labels.

In [ ]:
def get_predictions(
    model,
    loader
):

    model.eval()

    all_predictions = []
    all_labels = []


    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="Inference"
        ):

            images = images.to(device)

            outputs = model(
                images
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )


    return (
        all_labels,
        all_predictions
    )

## Calculate Accuracy, Precision, Recall, F1

In [ ]:
def get_predictions(
    model,
    loader
):

    model.eval()

    all_predictions = []
    all_labels = []


    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="Inference"
        ):

            images = images.to(device)

            outputs = model(
                images
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )


    return (
        all_labels,
        all_predictions
    )

results = {}

for name, model in trained_models.items():

    labels, predictions = get_predictions(
        model,
        test_loader
    )

    results[name] = {

        "Labels": labels,

        "Predictions": predictions
    }


metrics = {}

for name, result in results.items():

    labels = result["Labels"]
    predictions = result["Predictions"]


    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )


    metrics[name] = {

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-Score": f1
    }


comparison = pd.DataFrame(
    metrics
).T

comparison

## Final Comparison Table

In [ ]:
comparison_percent = (
    comparison * 100
).round(2)

comparison_percent

## Visualize Model Performance

In [ ]:
comparison.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Performance Comparison of Pre-trained Models"
)

plt.xlabel("Model")

plt.ylabel("Score")

plt.ylim(
    0,
    1
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.legend(
    title="Metrics"
)

plt.tight_layout()

plt.show()

## Confusion Matrices
The confusion matrix shows how the model's predictions are distributed across the different celebrity classes. The rows represent the actual classes, while the columns represent the predicted classes. Values along the diagonal represent correctly classified images, while values outside the diagonal indicate misclassifications between celebrity classes.

In [ ]:
for name, result in results.items():

    cm = confusion_matrix(
        result["Labels"],
        result["Predictions"]
    )

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=classes
    )

    display.plot(
        ax=ax,
        xticks_rotation=90,
        values_format="d"
    )

    plt.title(
        f"Confusion Matrix - {name}"
    )

    plt.tight_layout()

    plt.show()

## Model Complexity

The table presents the total and trainable number of parameters for each model. Total parameters provide an indication of the overall model complexity, while trainable parameters show the number of parameters updated during the custom classification task. Models with more parameters generally require more computational resources.

In [ ]:
parameter_data = []

for name, model in trained_models.items():

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    parameter_data.append({

        "Model": name,

        "Total Parameters":
            total_parameters,

        "Trainable Parameters":
            trainable_parameters
    })


parameter_df = pd.DataFrame(
    parameter_data
)

parameter_df

## Parameter Visualization
The bar chart compares the number of parameters across the five architectures. This visualization highlights the differences in model complexity and provides additional context when interpreting their classification performance.

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.bar(
    parameter_df["Model"],
    parameter_df["Total Parameters"]
)

plt.xlabel("Model")

plt.ylabel(
    "Number of Parameters"
)

plt.title(
    "Model Parameter Comparison"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

## Analysis of Results

The five models produced different classification results on the Celebrity Faces dataset. EfficientNet-B0 achieved the highest accuracy (60.74%), followed by MobileNetV3-Large (59.63%), ResNet50 (55.93%), DenseNet121 (51.85%), and ResNet18 (48.15%).

The results show that model architecture affects classification performance. EfficientNet-B0 and MobileNetV3-Large performed better while using fewer parameters than the larger ResNet models. Some differences between training and validation accuracy also suggest that the models may have experienced some overfitting.

## Conclusion

The experiment successfully demonstrated image classification using a custom Celebrity Faces dataset and five pre-trained PyTorch models. The models were evaluated using accuracy, precision, recall, and F1-score.

Among the tested models, EfficientNet-B0 achieved the highest overall test performance with 60.74% accuracy and a 60.22% F1-score. The results show that model architecture and complexity can affect performance when applying pre-trained models to a custom dataset.set.